# Monitor de Voos — v4 (calendário de 6 meses, teto de R$ 1.000)

**O que mudou em relação à v3**

| v3 | v4 |
|---|---|
| Datas fixas (28/10 a 05/11) | Varre **todas as datas de hoje até +180 dias** |
| Só `fast-flights`, 1 consulta por par de aeroportos | **`fli`** (Google Flights) com **multi-aeroporto numa só consulta** + `fast-flights` como reserva + **Aviasales/Travelpayouts** (opcional, gratuito) como radar |
| Alerta repetia o mesmo voo a cada execução | Alerta **só quando é novo ou quando o preço caiu** de novo |
| Mensagem curta, sem contexto | Mensagem com **dia da semana, horários, escalas, duração, folga até o teto, comparação com o histórico e o porquê do alerta** |
| Gráfico de "evolução" misturava viagens diferentes | Gráficos separam **data do voo** × **data da busca** (cada linha = um mês de viagem) |
| Token do Telegram escrito no código | Segredos lidos de **variáveis de ambiente / Kaggle Secrets / GitHub Secrets** |
| Só Kaggle, 1×/dia | Roda em Kaggle **ou** GitHub Actions (várias vezes ao dia) |

**Como funciona**

1. Para cada rota, consulta o voo **só ida** mais barato de cada data, nos dois sentidos (Alagoas/Sergipe → São Paulo e São Paulo → Alagoas/Sergipe).
2. Combina qualquer ida com qualquer volta que respeite a permanência mínima/máxima → acha a melhor **ida + volta** para cada data de saída (podendo ser cias e aeroportos diferentes).
3. Alerta no Telegram quando `preço da ida + preço da volta ≤ teto` (padrão R$ 1.000, estadia de 5 a 7 dias).
4. Grava o histórico no próprio repositório (Parquet em `dados/`) e gera um painel.

**Modos de execução**
- `completo`: varre os 180 dias (≈ 360 consultas por rota). Roda 1×/dia.
- `rapido`: reconsulta só as datas mais baratas da última varredura completa + os próximos 14 dias (≈ 60 consultas). Ideal para rodar a cada poucas horas sem chamar atenção do Google.
- `auto` (padrão): decide sozinho — `completo` se a última varredura completa tem mais de 20 h, senão `rapido`.

**Segredos necessários** (Kaggle → *Add-ons → Secrets*; GitHub → *Settings → Secrets and variables → Actions*)

| Nome | Obrigatório | Conteúdo |
|---|---|---|
| `TELEGRAM_TOKEN` | sim | token do bot (BotFather) |
| `TELEGRAM_CHAT_IDS` | sim | ids separados por vírgula, ex.: `111111111,222222222` (os seus ficam só no segredo) |
| `TRAVELPAYOUTS_TOKEN` | não | token gratuito do Travelpayouts (ativa a 2ª fonte) |


In [ ]:
# CÉLULA 1 — INSTALAÇÃO
# Instala só o que estiver faltando (no GitHub Actions o requirements.txt já resolve).
# A fli vem do GitHub (versão de setembro/2026), porque a versão do PyPI ainda usa
# um endpoint de calendário que o Google bloqueou em agosto/2026.
import importlib.util, subprocess, sys

def _instalar(pacote_import, spec_pip):
    if importlib.util.find_spec(pacote_import) is None:
        print(f"Instalando {spec_pip} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", spec_pip], check=False)

_instalar("fli", "flights @ git+https://github.com/punitarani/fli.git@881aee5ff4321e81ea2157cb44be94ce6a21dc1b")
_instalar("fast_flights", "fast-flights")
_instalar("pyarrow", "pyarrow")
print("Dependências OK.")


In [ ]:
# CÉLULA 2 — PARÂMETROS  (é a única célula que você precisa editar)
import os

# ---------------------------------------------------------------------------
# ROTAS MONITORADAS
# origens/destinos: códigos IATA. Todos os aeroportos de cada lado entram numa
# única consulta (o Google devolve o voo mais barato entre eles).
#   "ida"   = origens → destinos   (aqui: Alagoas/Sergipe → São Paulo)
#   "volta" = destinos → origens   (São Paulo → Alagoas/Sergipe)
# teto_ida_volta: alerta quando preço da ida + preço da volta <= este valor.
# min_noites / max_noites: dias entre o voo de ida e o de volta (estadia).
# teto_trecho:  (opcional) alerta também um trecho avulso <= este valor. None desliga.
# ---------------------------------------------------------------------------
ROTAS = [
    dict(
        nome="Alagoas ⇄ São Paulo",
        origens=["MCZ", "AJU"],
        destinos=["GRU", "CGH", "VCP"],
        teto_ida_volta=1000,
        min_noites=5,
        max_noites=7,
        teto_trecho=None,
    ),
]

HORIZONTE_DIAS = 180          # até quantos dias à frente procurar
ANTECEDENCIA_MIN_DIAS = 1     # não consulta voos para hoje
MAX_ESCALAS = 1               # 0 = só direto; 1 = até 1 escala; None = qualquer
PERMITIR_AEROPORTOS_DIFERENTES = True   # volta por aeroporto diferente da ida

# --- Fontes de preço ---------------------------------------------------------
FONTE = os.environ.get("FONTE", "auto")  # auto | fli | fast_flights | simulada
USAR_AVIASALES = True         # só é usado se existir o segredo TRAVELPAYOUTS_TOKEN
WORKERS = 4                   # consultas simultâneas (não passe de 6)
PAUSA_SEG = 0.4               # pausa por consulta, por worker (anti-bloqueio)

# --- Modo de execução --------------------------------------------------------
MODO = os.environ.get("MODO", "auto")   # auto | completo | rapido
HORAS_ENTRE_COMPLETAS = 20    # no modo auto: completo se a última completa for mais velha
TOP_DATAS_RAPIDO = 25         # no modo rápido: reconsulta as N datas mais baratas de cada sentido
DIAS_PROXIMOS_RAPIDO = 14     # ... e também todas as datas dos próximos N dias

# --- Alertas -----------------------------------------------------------------
MAX_OFERTAS_POR_MSG = 5
QUEDA_MIN_PARA_REALERTAR = 0.03   # só re-alerta a mesma viagem se cair >= 3%
ENVIAR_RESUMO_DIARIO = True       # resumo + painel após a varredura completa
DRY_RUN = os.environ.get("DRY_RUN", "0") == "1"   # 1 = imprime em vez de enviar

# --- Histórico ---------------------------------------------------------------
PASTA_DADOS = os.environ.get("PASTA_DADOS", "dados")   # histórico em Parquet dentro do repositório
FUSO = "America/Sao_Paulo"

print(f"{len(ROTAS)} rota(s) · horizonte {HORIZONTE_DIAS} dias · modo={MODO} · fonte={FONTE} · dry_run={DRY_RUN}")


In [ ]:
# CÉLULA 3 — UTILITÁRIOS (segredos, datas, formatação)
import json, time, hashlib, random, warnings
from dataclasses import dataclass, asdict, field
from datetime import date, datetime, timedelta
from zoneinfo import ZoneInfo
from urllib.parse import quote

import pandas as pd
import requests

warnings.filterwarnings("ignore", category=FutureWarning)
TZ = ZoneInfo(FUSO)
AGORA = datetime.now(TZ)
HOJE = AGORA.date()


def segredo(nome, padrao=None):
    """Lê um segredo de: variável de ambiente → Kaggle Secrets → Colab userdata."""
    v = os.environ.get(nome)
    if v:
        return v
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(nome)
    except Exception:
        pass
    try:
        from google.colab import userdata
        return userdata.get(nome)
    except Exception:
        pass
    return padrao


TELEGRAM_TOKEN = segredo("TELEGRAM_TOKEN")
TELEGRAM_CHAT_IDS = [c.strip() for c in (segredo("TELEGRAM_CHAT_IDS") or "").split(",") if c.strip()]
TRAVELPAYOUTS_TOKEN = segredo("TRAVELPAYOUTS_TOKEN")

DIAS_SEMANA = ["seg", "ter", "qua", "qui", "sex", "sáb", "dom"]
MESES = ["jan", "fev", "mar", "abr", "mai", "jun", "jul", "ago", "set", "out", "nov", "dez"]
CIAS = {"JJ": "LATAM", "LA": "LATAM", "G3": "GOL", "AD": "Azul", "2Z": "Voepass",
        "TP": "TAP", "AR": "Aerolíneas", "CM": "Copa"}


def brl(v):
    """1234.5 → 'R$ 1.235'"""
    if v is None or pd.isna(v):
        return "—"
    return "R$ " + f"{v:,.0f}".replace(",", ".")


def fmt_data(d, com_dia=True):
    d = pd.Timestamp(d).date()
    txt = f"{d.day:02d}/{d.month:02d}"
    return f"{DIAS_SEMANA[d.weekday()]} {txt}" if com_dia else txt


def fmt_dur(minutos):
    if minutos is None or pd.isna(minutos):
        return "—"
    minutos = int(minutos)
    return f"{minutos // 60}h{minutos % 60:02d}"


def fmt_escalas(n):
    if n is None or pd.isna(n):
        return "—"
    return "direto" if int(n) == 0 else f"{int(n)} escala" + ("s" if int(n) > 1 else "")


def link_google(origem, destino, d):
    """Busca genérica no Google Flights (usada quando a fonte não dá deep link)."""
    q = quote(f"Flights from {origem} to {destino} on {pd.Timestamp(d).date()} one way")
    return f"https://www.google.com/travel/flights?q={q}&curr=BRL&hl=pt-BR&gl=BR"


def datas_horizonte(dias_extras=0):
    ini = HOJE + timedelta(days=ANTECEDENCIA_MIN_DIAS)
    return [ini + timedelta(days=i) for i in range(HORIZONTE_DIAS + dias_extras)]


@dataclass
class Oferta:
    rota: str
    sentido: str          # "ida" (origens→destinos) ou "volta"
    data_voo: str         # YYYY-MM-DD
    preco: float
    cia: str
    origem: str
    destino: str
    escalas: int | None
    duracao_min: int | None
    partida: str          # HH:MM
    chegada: str          # HH:MM
    fonte: str
    link: str


print(f"Agora: {AGORA:%d/%m/%Y %H:%M} ({FUSO})")
print(f"Telegram: {len(TELEGRAM_CHAT_IDS)} destinatário(s) · "
      f"Travelpayouts: {'sim' if TRAVELPAYOUTS_TOKEN else 'não'}")


In [ ]:
# CÉLULA 4 — FONTES DE PREÇO
# Cada fonte recebe (origens, destinos, data) e devolve a Oferta mais barata ou None.
#   fli          → Google Flights, multi-aeroporto numa consulta, deep link do voo exato
#   fast_flights → Google Flights (biblioteca da v3), 1 consulta por par de aeroportos
#   simulada     → preços falsos e determinísticos, para testar o notebook sem internet
#   aviasales    → cache do Travelpayouts (1 chamada devolve o mês inteiro); radar extra
from concurrent.futures import ThreadPoolExecutor, as_completed


def _nome_cia(codigo, nome=None):
    return CIAS.get(codigo, nome or codigo)


def _por_cia(opcoes):
    """Mantém só a opção mais barata de cada companhia (dado extra gravado em dados/ofertas)."""
    melhores = {}
    for o in opcoes:
        if o["cia"] not in melhores or o["preco"] < melhores[o["cia"]]["preco"]:
            melhores[o["cia"]] = o
    return sorted(melhores.values(), key=lambda o: o["preco"])


# ---------------------------------------------------------------- fli
def buscar_fli(origens, destinos, d):
    from fli.models import (Airport, FlightSearchFilters, FlightSegment,
                            PassengerInfo, SeatType, MaxStops, SortBy)
    from fli.search import SearchFlights

    paradas = {0: MaxStops.NON_STOP, 1: MaxStops.ONE_STOP_OR_FEWER,
               2: MaxStops.TWO_OR_FEWER_STOPS}.get(MAX_ESCALAS, MaxStops.ANY)
    filtros = FlightSearchFilters(
        passenger_info=PassengerInfo(adults=1),
        flight_segments=[FlightSegment(
            departure_airport=[[Airport[o], 0] for o in origens],
            arrival_airport=[[Airport[x], 0] for x in destinos],
            travel_date=d.isoformat())],
        seat_type=SeatType.ECONOMY, stops=paradas, sort_by=SortBy.CHEAPEST)
    busca = SearchFlights()
    voos = busca.search(filtros, currency="BRL", language="pt-BR", country="BR")
    voos = [v for v in (voos or []) if v.price]
    if not voos:
        return None

    def resumo(v):
        l0, l1 = v.legs[0], v.legs[-1]
        return dict(preco=float(v.price),
                    cia=" + ".join(sorted({_nome_cia(l.airline.name, l.airline.value) for l in v.legs})),
                    origem=l0.departure_airport.name, destino=l1.arrival_airport.name,
                    escalas=int(v.stops), duracao_min=int(v.duration),
                    partida=l0.departure_datetime.strftime("%H:%M"),
                    chegada=l1.arrival_datetime.strftime("%H:%M"))

    v = min(voos, key=lambda x: x.price)
    melhor = resumo(v)
    try:
        melhor["link"] = busca.build_flight_booking_url(v, currency="BRL", language="pt-BR", country="BR")
    except Exception:
        melhor["link"] = link_google(melhor["origem"], melhor["destino"], d)
    melhor["extras"] = _por_cia(resumo(x) for x in voos)
    return melhor


# ---------------------------------------------------------------- fast-flights
def buscar_fast_flights(origens, destinos, d):
    from fast_flights import create_query, FlightQuery, Passengers, get_flights
    melhor, opcoes = None, []
    for o in origens:
        for x in destinos:
            q = create_query(
                flights=[FlightQuery(date=d.isoformat(), from_airport=o, to_airport=x)],
                trip="one-way", seat="economy", passengers=Passengers(adults=1),
                language="pt-BR", currency="BRL", max_stops=MAX_ESCALAS)
            try:
                res = get_flights(q)
            except Exception:
                continue
            for f in (res or []):
                if not f.price:
                    continue
                s0, s1 = f.flights[0], f.flights[-1]
                op = dict(preco=float(f.price), cia=" + ".join(_nome_cia(c) for c in f.airlines),
                          origem=s0.from_airport.code, destino=s1.to_airport.code,
                          escalas=max(0, len(f.flights) - 1),
                          duracao_min=sum(s.duration or 0 for s in f.flights),
                          partida="%02d:%02d" % tuple(s0.departure.time),
                          chegada="%02d:%02d" % tuple(s1.arrival.time))
                opcoes.append(op)
                if melhor is None or op["preco"] < melhor["preco"]:
                    melhor = dict(op, link=q.url())
            time.sleep(PAUSA_SEG)
    if melhor:
        melhor["extras"] = _por_cia(opcoes)
    return melhor


# ---------------------------------------------------------------- simulada
def buscar_simulada(origens, destinos, d):
    """Preço plausível e estável por data (varia um pouco a cada hora), sem internet."""
    semente = int(hashlib.md5(f"{origens}{destinos}{d}".encode()).hexdigest(), 16)
    rnd = random.Random(semente)
    if rnd.random() < 0.03:
        return None
    base = 430 + (180 if d.month in (12, 1, 7) else 0) + (60 if d.weekday() in (4, 6) else 0)
    antecedencia = (d - HOJE).days
    base += max(0, 25 - antecedencia) * 18            # perto da data fica caro
    ruido = random.Random(semente + AGORA.hour).uniform(-0.06, 0.06)
    preco = round(base * rnd.uniform(0.62, 1.45) * (1 + ruido))
    o, x = rnd.choice(origens), rnd.choice(destinos)
    h = rnd.randint(5, 21)
    dur = rnd.choice([180, 195, 240, 330, 410])
    cias = ["LATAM", "GOL", "Azul"]
    rnd.shuffle(cias)
    melhor = dict(preco=float(preco), cia=cias[0], origem=o, destino=x,
                  escalas=0 if dur < 200 else 1, duracao_min=dur, partida=f"{h:02d}:{rnd.choice([0, 15, 30, 45]):02d}",
                  chegada=f"{(h + dur // 60) % 24:02d}:{dur % 60:02d}")
    extras = [dict(melhor)] + [dict(melhor, cia=c, preco=float(round(preco * rnd.uniform(1.03, 1.6))))
                               for c in cias[1:]]
    return dict(melhor, link=link_google(o, x, d), extras=extras)


FONTES = {"fli": buscar_fli, "fast_flights": buscar_fast_flights, "simulada": buscar_simulada}


def escolher_fonte(rota):
    """Testa as fontes em ordem com 3 datas e fica com a primeira que responde."""
    if FONTE != "auto":
        return FONTE
    teste = [HOJE + timedelta(days=n) for n in (20, 45, 75)]
    for nome in ("fli", "fast_flights"):
        try:
            if any(FONTES[nome](rota["origens"], rota["destinos"], d) for d in teste):
                print(f"[fonte] {nome} respondeu — usando {nome}.")
                return nome
            print(f"[fonte] {nome} não trouxe preços nas datas de teste.")
        except Exception as e:
            print(f"[fonte] {nome} falhou: {type(e).__name__}: {str(e)[:120]}")
    raise RuntimeError("Nenhuma fonte do Google Flights respondeu (possível bloqueio).")


def _consultar(fonte, origens, destinos, d, tentativas=2):
    for t in range(tentativas):
        try:
            r = FONTES[fonte](origens, destinos, d)
            time.sleep(PAUSA_SEG)
            return r, None
        except Exception as e:
            erro = f"{type(e).__name__}: {str(e)[:100]}"
            time.sleep(1.5 * (t + 1))
    return None, erro


def varrer(rota, sentido, datas, fonte):
    """Consulta todas as datas em paralelo. Devolve (lista de Oferta, estatísticas)."""
    ori, dst = (rota["origens"], rota["destinos"]) if sentido == "ida" else (rota["destinos"], rota["origens"])
    ofertas, stats = [], {"consultas": len(datas), "sem_voo": 0, "falhas": 0, "erros": [], "extras": []}
    with ThreadPoolExecutor(max_workers=WORKERS) as ex:
        futuros = {ex.submit(_consultar, fonte, ori, dst, d): d for d in datas}
        for i, fut in enumerate(as_completed(futuros), 1):
            d = futuros[fut]
            r, erro = fut.result()
            if erro:
                stats["falhas"] += 1
                stats["erros"].append(erro)
            elif r is None:
                stats["sem_voo"] += 1
            else:
                for e in r.pop("extras", []):
                    stats["extras"].append(dict(e, rota=rota["nome"], sentido=sentido,
                                                data_voo=d.isoformat(), fonte=fonte))
                ofertas.append(Oferta(rota=rota["nome"], sentido=sentido, data_voo=d.isoformat(),
                                      fonte=fonte, **r))
            if i % 40 == 0 or i == len(datas):
                print(f"  {sentido}: {i}/{len(datas)} consultas · {len(ofertas)} com preço · {stats['falhas']} falhas")
    return ofertas, stats


# ---------------------------------------------------------------- aviasales (radar)
def buscar_aviasales(rota, sentido, meses):
    """Preço mais barato por dia, mês a mês, do cache do Aviasales (pode ter até 48 h)."""
    if not (USAR_AVIASALES and TRAVELPAYOUTS_TOKEN):
        return []
    ori, dst = (rota["origens"], rota["destinos"]) if sentido == "ida" else (rota["destinos"], rota["origens"])
    melhores = {}
    for o in ori:
        for x in dst:
            for mes in meses:
                try:
                    r = requests.get(
                        "https://api.travelpayouts.com/aviasales/v3/grouped_prices",
                        params=dict(origin=o, destination=x, departure_at=mes, group_by="departure_at",
                                    currency="brl", market="br", direct=str(MAX_ESCALAS == 0).lower()),
                        headers={"X-Access-Token": TRAVELPAYOUTS_TOKEN, "Accept-Encoding": "gzip"},
                        timeout=20)
                    dados = r.json().get("data") or {}
                except Exception as e:
                    print(f"  [aviasales] {o}->{x} {mes}: {e}")
                    continue
                for dia, it in dados.items():
                    if MAX_ESCALAS is not None and (it.get("transfers") or 0) > MAX_ESCALAS:
                        continue
                    preco = float(it["price"])
                    if dia not in melhores or preco < melhores[dia].preco:
                        dep = pd.Timestamp(it["departure_at"])
                        dur = it.get("duration_to") or it.get("duration")
                        melhores[dia] = Oferta(
                            rota=rota["nome"], sentido=sentido, data_voo=dia, preco=preco,
                            cia=_nome_cia(it.get("airline", "?")),
                            origem=it.get("origin_airport", o), destino=it.get("destination_airport", x),
                            escalas=it.get("transfers"), duracao_min=dur, partida=dep.strftime("%H:%M"),
                            chegada=(dep + pd.Timedelta(minutes=dur or 0)).strftime("%H:%M") if dur else "—",
                            fonte="aviasales", link="https://www.aviasales.com" + it.get("link", ""))
                time.sleep(0.15)
    return list(melhores.values())

print("Fontes prontas:", ", ".join(FONTES) + (" + aviasales" if TRAVELPAYOUTS_TOKEN else ""))


In [ ]:
# CÉLULA 5 — HISTÓRICO NO PRÓPRIO REPOSITÓRIO (Parquet particionado)
#
# dados/
#   precos/AAAA/MM/<execucao>.parquet    menor preço por (rota, sentido, data do voo) em cada execução
#   ofertas/AAAA/MM/<execucao>.parquet   menor preço de CADA companhia por data (mais detalhe p/ análise)
#   alertas/AAAA/MM/<execucao>.parquet   ofertas enviadas no Telegram (evita repetir)
#   execucoes.csv                        1 linha por execução: modo, fonte, falhas, melhor preço, duração...
#   ultimo/                              painel e tabelas da última varredura completa (sobrescritos)
#
# Cada execução cria arquivos NOVOS e pequenos (append-only): o git guarda só o que mudou,
# o repositório cresce devagar e dá para consultar tudo com pandas ou DuckDB:
#   duckdb.sql("select * from 'dados/precos/**/*.parquet'")
import glob, pathlib

ID_EXECUCAO = AGORA.strftime("%Y%m%dT%H%M%S")
COLS_PRECOS = ["data_busca", "modo"] + list(Oferta.__dataclass_fields__)
COLS_OFERTAS = ["data_busca", "rota", "sentido", "data_voo", "cia", "preco", "origem", "destino",
                "escalas", "duracao_min", "partida", "chegada", "fonte"]
COLS_ALERTAS = ["enviado_em", "rota", "data_ida", "data_volta", "total", "preco_ida", "preco_volta",
                "cia_ida", "cia_volta"]
TIPOS = {"preco": "float64", "total": "float64", "preco_ida": "float64", "preco_volta": "float64",
         "escalas": "float64", "duracao_min": "float64"}


class Historico:
    def __init__(self, pasta=PASTA_DADOS):
        self.raiz = pathlib.Path(pasta)
        self.raiz.mkdir(parents=True, exist_ok=True)

    def _arquivo(self, tabela):
        p = self.raiz / tabela / AGORA.strftime("%Y") / AGORA.strftime("%m") / f"{ID_EXECUCAO}.parquet"
        p.parent.mkdir(parents=True, exist_ok=True)
        return p

    def _gravar(self, tabela, colunas, df):
        if df is None or df.empty:
            return
        df = df.reindex(columns=colunas).copy()
        for c in colunas:
            if c in TIPOS:
                df[c] = pd.to_numeric(df[c], errors="coerce").astype(TIPOS[c])
            elif c in ("data_busca", "enviado_em"):
                df[c] = pd.to_datetime(df[c], utc=True)
            elif c.startswith("data_"):
                df[c] = pd.to_datetime(df[c]).dt.date
            else:
                df[c] = df[c].astype("string")
        df.to_parquet(self._arquivo(tabela), index=False)

    def _ler(self, tabela, colunas, desde=None):
        arquivos = sorted(glob.glob(str(self.raiz / tabela / "*" / "*" / "*.parquet")))
        if desde is not None:   # nome do arquivo = AAAAMMDDTHHMMSS → filtra sem abrir
            corte = desde.strftime("%Y%m%dT%H%M%S")
            arquivos = [a for a in arquivos if pathlib.Path(a).stem >= corte]
        if not arquivos:
            return pd.DataFrame(columns=colunas)
        return pd.concat([pd.read_parquet(a) for a in arquivos], ignore_index=True)

    # ---- alto nível
    def ler_precos(self):
        df = self._ler("precos", COLS_PRECOS)
        if df.empty:
            return df
        df["data_busca"] = pd.to_datetime(df["data_busca"], utc=True).dt.tz_convert(TZ)
        df["data_voo"] = pd.to_datetime(df["data_voo"]).dt.date
        return df

    def gravar_precos(self, df):
        self._gravar("precos", COLS_PRECOS, df)

    def gravar_ofertas(self, df):
        self._gravar("ofertas", COLS_OFERTAS, df)

    def ler_alertas(self):
        df = self._ler("alertas", COLS_ALERTAS)
        if not df.empty:
            df["data_ida"] = df["data_ida"].astype(str)
            df["data_volta"] = df["data_volta"].astype(str)
        return df

    def gravar_alertas(self, df):
        self._gravar("alertas", COLS_ALERTAS, df)

    def registrar_execucao(self, linha: dict):
        caminho = self.raiz / "execucoes.csv"
        novo = pd.DataFrame([linha])
        novo.to_csv(caminho, mode="a", index=False, header=not caminho.exists())

    def salvar_ultimo(self, nome, conteudo_df=None, arquivo_origem=None):
        pasta = self.raiz / "ultimo"
        pasta.mkdir(exist_ok=True)
        if conteudo_df is not None:
            conteudo_df.to_csv(pasta / nome, index=False)
        if arquivo_origem is not None:
            import shutil
            shutil.copy(arquivo_origem, pasta / nome)


hist = Historico()
df_hist = hist.ler_precos()
df_alertas_ant = hist.ler_alertas()
n_exec = df_hist["data_busca"].nunique() if not df_hist.empty else 0
print(f"Histórico em '{PASTA_DADOS}/': {len(df_hist)} preços de {n_exec} execuções · "
      f"{len(df_alertas_ant)} alertas já enviados")


In [ ]:
# CÉLULA 6 — VARREDURA (decide o modo, escolhe a fonte e consulta as datas)

def ultima_completa(rota_nome):
    if df_hist.empty:
        return None
    h = df_hist[(df_hist["rota"] == rota_nome) & (df_hist["modo"] == "completo")]
    return None if h.empty else h["data_busca"].max()


def decidir_modo(rota_nome):
    if MODO in ("completo", "rapido"):
        return MODO if (MODO == "completo" or ultima_completa(rota_nome) is not None) else "completo"
    t = ultima_completa(rota_nome)
    if t is None or (AGORA - t) > timedelta(hours=HORAS_ENTRE_COMPLETAS):
        return "completo"
    return "rapido"


def datas_para(rota_nome, sentido, modo, extra=0):
    todas = datas_horizonte(extra if sentido == "volta" else 0)
    if modo == "completo":
        return todas
    t = ultima_completa(rota_nome)
    h = df_hist[(df_hist["rota"] == rota_nome) & (df_hist["sentido"] == sentido) &
                (df_hist["data_busca"] == t) & (df_hist["data_voo"] >= todas[0])]
    baratas = set(h.nsmallest(TOP_DATAS_RAPIDO, "preco")["data_voo"])
    proximas = set(todas[:DIAS_PROXIMOS_RAPIDO])
    return sorted(baratas | proximas)


ofertas_execucao, estatisticas, modos = [], {}, {}
for rota in ROTAS:
    nome = rota["nome"]
    modo = decidir_modo(nome)
    modos[nome] = modo
    print(f"\n=== {nome} · modo {modo} ===")
    fonte = escolher_fonte(rota)
    for sentido in ("ida", "volta"):
        datas = datas_para(nome, sentido, modo, extra=rota["max_noites"])
        if not datas:
            estatisticas[(nome, sentido)] = {"consultas": 0, "sem_voo": 0, "falhas": 0, "erros": [], "extras": [], "fonte": fonte}
            continue
        print(f"{sentido}: {len(datas)} datas ({fmt_data(datas[0], False)} a {fmt_data(datas[-1], False)})")
        ofs, st = varrer(rota, sentido, datas, fonte)
        st["fonte"] = fonte
        if modo == "completo":
            meses = sorted({d.strftime("%Y-%m") for d in datas})
            radar = buscar_aviasales(rota, sentido, meses)
            st["aviasales"] = len(radar)
            ofs += radar
        estatisticas[(nome, sentido)] = st
        ofertas_execucao += ofs

df_atual = pd.DataFrame([asdict(o) for o in ofertas_execucao])
if df_atual.empty:
    df_atual = pd.DataFrame(columns=COLS_PRECOS)
df_atual["data_busca"] = AGORA
df_atual["modo"] = df_atual["rota"].map(modos)
df_atual["data_voo"] = pd.to_datetime(df_atual["data_voo"]).dt.date

# dado extra: a opção mais barata de cada companhia, em cada data consultada
df_ofertas = pd.DataFrame([e for st in estatisticas.values() for e in st["extras"]])
df_ofertas["data_busca"] = AGORA

print(f"\n{len(df_atual)} preços coletados nesta execução · {len(df_ofertas)} opções por companhia.")
for (r, s), st in estatisticas.items():
    print(f"  {r} / {s}: {st['consultas']} consultas · {st['sem_voo']} sem voo · {st['falhas']} falhas"
          + (f" · {st.get('aviasales', 0)} datas do Aviasales" if 'aviasales' in st else ""))


In [ ]:
# CÉLULA 7 — SALVA OS PREÇOS DESTA EXECUÇÃO NO HISTÓRICO
# (antes dos alertas: se o Telegram falhar, o histórico já está garantido)
hist.gravar_precos(df_atual)
hist.gravar_ofertas(df_ofertas)
df_hist = pd.concat([df for df in (df_hist, df_atual[COLS_PRECOS]) if not df.empty], ignore_index=True)
df_hist["data_busca"] = pd.to_datetime(df_hist["data_busca"], utc=True).dt.tz_convert(TZ)
for c in ("preco", "escalas", "duracao_min"):
    df_hist[c] = pd.to_numeric(df_hist[c], errors="coerce")
print(f"Histórico agora com {len(df_hist)} linhas.")


In [ ]:
# CÉLULA 8 — MONTA AS MELHORES IDA+VOLTA E O CONTEXTO DE CADA OFERTA

def tabela_vigente(rota_nome, ate=None, janela_h=30):
    """Preço mais recente de cada (sentido, data do voo), usando buscas das últimas `janela_h` horas.
    No modo rápido, as datas não reconsultadas usam o preço da última varredura completa."""
    ate = ate or AGORA
    h = df_hist[(df_hist["rota"] == rota_nome) & (df_hist["data_busca"] <= ate) &
                (df_hist["data_busca"] > ate - timedelta(hours=janela_h))].copy()
    if h.empty:
        return h
    h = h[h["data_voo"] > pd.Timestamp(ate).date()]
    h = (h.sort_values("data_busca").drop_duplicates(["sentido", "data_voo", "fonte"], keep="last")
          .sort_values("preco").drop_duplicates(["sentido", "data_voo"], keep="first"))
    h["idade_h"] = (ate - h["data_busca"]).dt.total_seconds() / 3600
    return h.reset_index(drop=True)


def combinar(tab, rota):
    """Todas as combinações ida × volta válidas → a melhor volta para cada data de ida."""
    ida, volta = tab[tab["sentido"] == "ida"], tab[tab["sentido"] == "volta"]
    if ida.empty or volta.empty:
        return pd.DataFrame()
    m = ida.add_suffix("_ida").merge(volta.add_suffix("_volta"), how="cross")
    m["noites"] = (pd.to_datetime(m["data_voo_volta"]) - pd.to_datetime(m["data_voo_ida"])).dt.days
    m = m[m["noites"].between(rota["min_noites"], rota["max_noites"])]
    if not PERMITIR_AEROPORTOS_DIFERENTES:
        m = m[(m["destino_ida"] == m["origem_volta"]) & (m["origem_ida"] == m["destino_volta"])]
    m["total"] = m["preco_ida"] + m["preco_volta"]
    m = m.sort_values("total").drop_duplicates("data_voo_ida").sort_values("data_voo_ida")
    return m.reset_index(drop=True)


def contexto_trecho(rota_nome, sentido, d, preco_agora):
    """Compara o preço de hoje com o histórico da MESMA data de voo."""
    h = df_hist[(df_hist["rota"] == rota_nome) & (df_hist["sentido"] == sentido) &
                (df_hist["data_voo"] == d) & (df_hist["data_busca"] < AGORA)]
    if h.empty:
        return dict(min_ant=None, quando_min=None, primeira_vez=True, var_24h=None)
    i = h["preco"].idxmin()
    ontem = h[h["data_busca"] <= AGORA - timedelta(hours=20)]
    ref = ontem.sort_values("data_busca")["preco"].iloc[-1] if not ontem.empty else None
    return dict(min_ant=h.loc[i, "preco"], quando_min=h.loc[i, "data_busca"], primeira_vez=False,
                var_24h=None if ref is None else preco_agora - ref)


tabelas, combos, ofertas_teto, trechos_teto = {}, {}, {}, {}
for rota in ROTAS:
    nome = rota["nome"]
    tab = tabela_vigente(nome)
    tabelas[nome] = tab
    cb = combinar(tab, rota)
    combos[nome] = cb
    if cb.empty:
        print(f"{nome}: sem combinações válidas.")
        continue
    tipico = cb["total"].median()
    abaixo = cb[cb["total"] <= rota["teto_ida_volta"]].copy()
    abaixo["tipico"] = tipico
    abaixo["ctx_ida"] = [contexto_trecho(nome, "ida", r.data_voo_ida, r.preco_ida) for r in abaixo.itertuples()]
    abaixo["ctx_volta"] = [contexto_trecho(nome, "volta", r.data_voo_volta, r.preco_volta) for r in abaixo.itertuples()]
    ofertas_teto[nome] = abaixo.sort_values("total")
    if rota.get("teto_trecho"):
        trechos_teto[nome] = tab[tab["preco"] <= rota["teto_trecho"]].sort_values("preco")
    print(f"{nome}: {len(cb)} datas de ida com volta possível · melhor {brl(cb['total'].min())} · "
          f"mediana {brl(tipico)} · {len(abaixo)} abaixo do teto de {brl(rota['teto_ida_volta'])}")

if ofertas_teto:
    nome0 = next(iter(ofertas_teto))
    cols = ["data_voo_ida", "data_voo_volta", "noites", "origem_ida", "destino_ida", "origem_volta",
            "destino_volta", "cia_ida", "cia_volta", "preco_ida", "preco_volta", "total"]
    display(ofertas_teto[nome0][cols].head(10)) if len(ofertas_teto[nome0]) else None


In [ ]:
# CÉLULA 9 — PAINEL (3 gráficos, um por pergunta)
#  A) "Qual dia está mais barato?"          → preço só ida por DATA DO VOO (busca de agora)
#  B) "Qual ida+volta passa no teto?"       → melhor ida+volta por DATA DE SAÍDA, com a linha do teto
#  C) "Os preços estão caindo ou subindo?"  → menor ida+volta por MÊS DA VIAGEM ao longo das BUSCAS
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.ticker import FuncFormatter

COR_IDA, COR_VOLTA, COR_BOA, COR_TETO = "#2a78d6", "#eb6834", "#008300", "#52514e"
SERIES_MES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
FMT_BRL = FuncFormatter(lambda v, _: brl(v))
plt.rcParams.update({"figure.dpi": 110, "font.size": 10, "axes.titlesize": 12, "axes.titleweight": "bold",
                     "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#e6e5e0", "grid.linewidth": 0.8, "axes.edgecolor": "#8a8984"})


def _eixo_datas(ax):
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    ax.xaxis.set_minor_locator(mdates.WeekdayLocator(byweekday=mdates.MO))
    ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{MESES[mdates.num2date(v).month - 1]}/{mdates.num2date(v):%y}"))
    ax.yaxis.set_major_formatter(FMT_BRL)


def evolucao_por_mes(rota):
    """Para cada dia de busca: menor ida+volta por mês de saída (só com dados daquele dia)."""
    nome = rota["nome"]
    h = df_hist[df_hist["rota"] == nome]
    if h.empty:
        return pd.DataFrame()
    linhas = []
    for dia in sorted(h["data_busca"].dt.date.unique())[-45:]:
        fim = datetime.combine(dia, datetime.max.time(), TZ)
        cb = combinar(tabela_vigente(nome, ate=min(fim, AGORA), janela_h=30), rota)
        if cb.empty:
            continue
        cb["mes"] = pd.to_datetime(cb["data_voo_ida"]).dt.to_period("M")
        for mes, g in cb.groupby("mes"):
            linhas.append(dict(dia=pd.Timestamp(dia), mes=mes, total=g["total"].min()))
    return pd.DataFrame(linhas)


def desenhar_painel(rota, arquivo):
    nome, teto = rota["nome"], rota["teto_ida_volta"]
    tab, cb = tabelas[nome], combos[nome]
    fig, axs = plt.subplots(3, 1, figsize=(12, 13), constrained_layout=True,
                            gridspec_kw={"height_ratios": [1, 1, 0.9]})
    fig.suptitle(f"{nome} — busca de {AGORA:%d/%m/%Y %H:%M} · 1 adulto, econômica",
                 fontsize=14, fontweight="bold")

    # A) preço só ida por data de voo
    ax = axs[0]
    for sentido, cor, rot in (("ida", COR_IDA, f"Ida ({'/'.join(rota['origens'])} → {'/'.join(rota['destinos'])})"),
                              ("volta", COR_VOLTA, f"Volta ({'/'.join(rota['destinos'])} → {'/'.join(rota['origens'])})")):
        s = tab[tab["sentido"] == sentido].sort_values("data_voo")
        if s.empty:
            continue
        x = pd.to_datetime(s["data_voo"])
        i = s["preco"].idxmin()
        rot += f" — mín. {brl(s.loc[i, 'preco'])} em {fmt_data(s.loc[i, 'data_voo'])}"
        ax.plot(x, s["preco"], color=cor, lw=1.6, label=rot)
        ax.scatter(pd.to_datetime(s.loc[i, "data_voo"]), s.loc[i, "preco"], s=70, color=cor,
                   edgecolor="white", lw=2, zorder=3)
    if rota.get("teto_trecho"):
        ax.axhline(rota["teto_trecho"], ls="--", color=COR_TETO, lw=1.2)
        ax.text(1, rota["teto_trecho"], f" teto por trecho {brl(rota['teto_trecho'])}", transform=ax.get_yaxis_transform(),
                va="bottom", ha="right", fontsize=9, color=COR_TETO)
    ax.set_title("A · Preço do trecho só ida, por data do voo")
    ax.set_ylabel("Preço do trecho")
    ax.legend(loc="upper left", frameon=False)
    _eixo_datas(ax)

    # B) melhor ida+volta por data de saída
    ax = axs[1]
    if not cb.empty:
        x = pd.to_datetime(cb["data_voo_ida"])
        j = cb["total"].idxmin()
        ax.plot(x, cb["total"], color=COR_IDA, lw=1.6,
                label=(f"Melhor ida+volta saindo neste dia — melhor de todos: {brl(cb.loc[j, 'total'])} "
                       f"({fmt_data(cb.loc[j, 'data_voo_ida'])} → {fmt_data(cb.loc[j, 'data_voo_volta'])}, "
                       f"{int(cb.loc[j, 'noites'])} noites)"))
        ok = cb["total"] <= teto
        ax.scatter(x[ok], cb.loc[ok, "total"], s=40, color=COR_BOA, edgecolor="white", lw=1.5, zorder=3,
                   label=f"✓ dentro do teto ({ok.sum()} de {len(cb)} datas)")
    ax.axhline(teto, ls="--", color=COR_TETO, lw=1.2)
    ax.text(1, teto, f" teto {brl(teto)}", transform=ax.get_yaxis_transform(), va="bottom", ha="right",
            fontsize=9, color=COR_TETO)
    ax.set_title(f"B · Melhor ida+volta por data de saída ({rota['min_noites']} a {rota['max_noites']} noites)")
    ax.set_ylabel("Total ida+volta")
    ax.legend(loc="upper left", frameon=False)
    _eixo_datas(ax)

    # C) evolução ao longo das buscas
    ax = axs[2]
    ev = evolucao_por_mes(rota)
    if ev.empty or ev["dia"].nunique() < 2:
        ax.text(0.5, 0.5, "A evolução aparece a partir da 2ª busca em dias diferentes.",
                ha="center", va="center", transform=ax.transAxes, color="#52514e", fontsize=11)
        ax.set_xticks([]); ax.set_yticks([])
    else:
        for k, (mes, g) in enumerate(ev.groupby("mes")):
            g = g.sort_values("dia")
            cor = SERIES_MES[k % len(SERIES_MES)]
            rot = f"{MESES[mes.month - 1]}/{str(mes.year)[2:]} — hoje {brl(g['total'].iloc[-1])}"
            ax.plot(g["dia"], g["total"], color=cor, lw=2, marker="o", ms=4, label=rot)
        ax.axhline(teto, ls="--", color=COR_TETO, lw=1.2)
        ax.text(1, teto, f" teto {brl(teto)}", transform=ax.get_yaxis_transform(), va="bottom", ha="right",
                fontsize=9, color=COR_TETO)
        n_dias = ev["dia"].nunique()
        ax.xaxis.set_major_locator(mdates.DayLocator(interval=max(1, n_dias // 8)))
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%d/%m"))
        ax.yaxis.set_major_formatter(FMT_BRL)
        ax.legend(loc="upper left", bbox_to_anchor=(1.0, 1.0), frameon=False, title="Saída em")
    ax.set_title("C · Menor ida+volta encontrado em cada dia de busca, por mês da viagem")
    ax.set_xlabel("Dia em que a busca foi feita")
    ax.set_ylabel("Menor ida+volta")

    fig.savefig(arquivo, dpi=110, bbox_inches="tight")
    plt.show()
    return arquivo


paineis = {}
for i, rota in enumerate(ROTAS):
    if not combos[rota["nome"]].empty:
        paineis[rota["nome"]] = desenhar_painel(rota, f"painel_{i}.png")


In [ ]:
# CÉLULA 10 — MENSAGENS NO TELEGRAM
import html

TG_API = f"https://api.telegram.org/bot{TELEGRAM_TOKEN}"
NOME_MODO = {"completo": "varredura completa", "rapido": "checagem rápida"}


def _tg_post(metodo, n, **kw):
    """Chama a API do Telegram sem nunca imprimir o token nem o chat id (o repositório é público).
    Erros mostram só o nº do destinatário (1º, 2º...) e o código HTTP."""
    try:
        r = requests.post(f"{TG_API}/{metodo}", timeout=60, **kw)
        if r.status_code == 200:
            return True
            desc = r.json().get("description", "") if r.headers.get("content-type", "").startswith("application/json") else ""
            print(f"[ERRO TELEGRAM destinatário {n}] HTTP {r.status_code} {desc[:120]}")
    except Exception as e:                      # a mensagem da exceção contém a URL com o token
        print(f"[ERRO TELEGRAM destinatário {n}] {type(e).__name__} (detalhes ocultos)")
    return False


def tg_texto(texto):
    if DRY_RUN or not TELEGRAM_TOKEN or not TELEGRAM_CHAT_IDS:
        print("—" * 60 + "\n[TELEGRAM — não enviado]\n" + texto + "\n" + "—" * 60)
        return False
    partes, atual = [], ""
    for bloco in texto.split("\n\n"):            # respeita o limite de 4096 caracteres
        if len(atual) + len(bloco) + 2 > 3900:
            partes.append(atual); atual = ""
        atual += ("\n\n" if atual else "") + bloco
    partes.append(atual)
    algum_ok = False
    for n, cid in enumerate(TELEGRAM_CHAT_IDS, 1):
        for p in partes:
            algum_ok |= _tg_post("sendMessage", n, data=dict(chat_id=cid, text=p, parse_mode="HTML",
                                                              disable_web_page_preview=True))
    return algum_ok


def tg_foto(caminho, legenda):
    if DRY_RUN or not TELEGRAM_TOKEN or not TELEGRAM_CHAT_IDS:
        print(f"[TELEGRAM foto — não enviada] {caminho}: {legenda}")
        return
    for n, cid in enumerate(TELEGRAM_CHAT_IDS, 1):
        with open(caminho, "rb") as f:
            _tg_post("sendPhoto", n, files={"photo": f},
                     data=dict(chat_id=cid, caption=legenda[:1000], parse_mode="HTML"))


def linha_trecho(rotulo, emoji, d, o, x, cia, partida, chegada, escalas, dur, preco, fonte, link, idade_h):
    extra = " <i>(cache Aviasales — confirme)</i>" if fonte == "aviasales" else ""
    if idade_h and idade_h > 2:
        extra += f" <i>(preço de {idade_h:.0f} h atrás)</i>"
    return (f"{emoji} <b>{rotulo}</b> {fmt_data(d)} · {o}→{x} · {html.escape(str(cia))}\n"
            f"     {partida}→{chegada} · {fmt_escalas(escalas)} · {fmt_dur(dur)} · <b>{brl(preco)}</b>{extra}\n"
            f"     <a href=\"{html.escape(link, quote=True)}\">abrir este voo</a>")


def motivos_oferta(r, rota):
    m = [f"cabe no teto com folga de {brl(rota['teto_ida_volta'] - r.total)}"]
    pct = r.total / r.tipico - 1
    if pct <= -0.05:
        m.append(f"{abs(pct):.0%} abaixo da mediana das próximas datas ({brl(r.tipico)})")
    for rot, ctx, preco, d in (("ida", r.ctx_ida, r.preco_ida, r.data_voo_ida),
                               ("volta", r.ctx_volta, r.preco_volta, r.data_voo_volta)):
        if ctx["min_ant"] is not None and preco < ctx["min_ant"]:
            m.append(f"{rot} no menor preço já visto para {fmt_data(d, False)} (antes {brl(ctx['min_ant'])})")
        elif ctx["var_24h"] is not None and abs(ctx["var_24h"]) >= 10:
            verbo = "caiu" if ctx["var_24h"] < 0 else "subiu"
            m.append(f"{rot} {verbo} {brl(abs(ctx['var_24h']))} desde ontem")
    return m


def bloco_oferta(k, r, rota):
    avisos = []
    if r.destino_ida != r.origem_volta:
        avisos.append(f"chega em {r.destino_ida} e volta saindo de {r.origem_volta}")
    if r.origem_ida != r.destino_volta:
        avisos.append(f"sai de {r.origem_ida} e retorna para {r.destino_volta}")
    if r.cia_ida != r.cia_volta:
        avisos.append("cias diferentes = 2 reservas separadas")
    txt = (f"<b>{k}) {brl(r.total)} ida+volta</b> · {int(r.noites)} noites "
           f"({fmt_data(r.data_voo_ida)} → {fmt_data(r.data_voo_volta)}) · {(r.data_voo_ida - HOJE).days} dias até a viagem\n"
           + linha_trecho("Ida", "🛫", r.data_voo_ida, r.origem_ida, r.destino_ida, r.cia_ida, r.partida_ida,
                          r.chegada_ida, r.escalas_ida, r.duracao_min_ida, r.preco_ida, r.fonte_ida, r.link_ida, r.idade_h_ida) + "\n"
           + linha_trecho("Volta", "🛬", r.data_voo_volta, r.origem_volta, r.destino_volta, r.cia_volta, r.partida_volta,
                          r.chegada_volta, r.escalas_volta, r.duracao_min_volta, r.preco_volta, r.fonte_volta, r.link_volta, r.idade_h_volta)
           + "\n💡 <i>Por que avisei:</i> " + "; ".join(motivos_oferta(r, rota)) + ".")
    if avisos:
        txt += "\n⚠️ " + "; ".join(avisos) + "."
    return txt


RODAPE = ("<i>Preços para 1 adulto, econômica, só bagagem de mão. Ida e volta são compradas separadamente "
          "(por isso podem ser de cias diferentes). O preço muda rápido: confira no link antes de pagar.</i>")


def filtrar_novas(nome, df):
    """Remove viagens já alertadas, a não ser que o preço tenha caído pelo menos QUEDA_MIN_PARA_REALERTAR."""
    ant = df_alertas_ant[df_alertas_ant["rota"] == nome]
    ultimo = ant.groupby(["data_ida", "data_volta"])["total"].min().to_dict() if not ant.empty else {}
    novas = []
    for r in df.itertuples():
        prev = ultimo.get((str(r.data_voo_ida), str(r.data_voo_volta)))
        if prev is None or r.total <= prev * (1 - QUEDA_MIN_PARA_REALERTAR):
            novas.append(r)
    return novas


registro_alertas = []
for rota in ROTAS:
    nome = rota["nome"]
    est = [estatisticas[(nome, s)] for s in ("ida", "volta")]
    n_consultas = sum(e["consultas"] for e in est)
    n_falhas = sum(e["falhas"] for e in est)
    cab = (f"<i>{NOME_MODO[modos[nome]]} · {AGORA:%d/%m %H:%M} · {n_consultas} datas consultadas "
           f"via {est[0]['fonte']}</i>")

    # 1) Saúde: muitas falhas = provável bloqueio
    if n_consultas and n_falhas / n_consultas > 0.5:
        erros = pd.Series(sum((e["erros"] for e in est), [])).value_counts().head(2)
        tg_texto(f"⚠️ <b>Monitor com problema — {nome}</b>\n{cab}\n{n_falhas} de {n_consultas} consultas falharam. "
                 f"Erros mais comuns:\n<code>{html.escape(chr(10).join(erros.index))}</code>\n"
                 "Se repetir, o Google pode estar bloqueando o IP: aumente PAUSA_SEG ou reduza WORKERS.")

    # 2) Ofertas abaixo do teto (só as novas ou que baratearam)
    df_of = ofertas_teto.get(nome, pd.DataFrame())
    novas = filtrar_novas(nome, df_of) if len(df_of) else []
    if novas:
        msg = [f"✈️ <b>{len(novas)} ida+volta até {brl(rota['teto_ida_volta'])} — {nome}</b>\n{cab}"]
        for k, r in enumerate(novas[:MAX_OFERTAS_POR_MSG], 1):
            msg.append(bloco_oferta(k, r, rota))
        if len(novas) > MAX_OFERTAS_POR_MSG:
            resto = [fmt_data(d, False) for d in sorted({r.data_voo_ida for r in novas[MAX_OFERTAS_POR_MSG:]})]
            msg.append(f"…e mais {len(novas) - MAX_OFERTAS_POR_MSG} datas de saída abaixo do teto: {', '.join(resto[:15])}.")
        msg.append(RODAPE)
        enviado = tg_texto("\n\n".join(msg))
        if enviado:   # só marca como alertado se chegou a pelo menos um destinatário
            registro_alertas += [dict(enviado_em=AGORA, rota=nome, data_ida=str(r.data_voo_ida),
                                      data_volta=str(r.data_voo_volta), total=r.total,
                                      preco_ida=r.preco_ida, preco_volta=r.preco_volta,
                                      cia_ida=r.cia_ida, cia_volta=r.cia_volta) for r in novas]
    else:
        print(f"{nome}: nenhuma oferta nova abaixo do teto.")

    # 3) Trechos avulsos (se teto_trecho estiver ligado)
    tt = trechos_teto.get(nome)
    if tt is not None and len(tt):
        linhas = [f"🎯 <b>Trechos só ida até {brl(rota['teto_trecho'])} — {nome}</b>"]
        for r in tt.head(MAX_OFERTAS_POR_MSG).itertuples():
            linhas.append(linha_trecho(r.sentido.capitalize(), "🛫" if r.sentido == "ida" else "🛬", r.data_voo,
                                       r.origem, r.destino, r.cia, r.partida, r.chegada, r.escalas,
                                       r.duracao_min, r.preco, r.fonte, r.link, r.idade_h))
        tg_texto("\n\n".join(linhas))

    # 4) Resumo diário + painel (só depois da varredura completa)
    cb = combos[nome]
    if ENVIAR_RESUMO_DIARIO and modos[nome] == "completo" and not cb.empty:
        teto = rota["teto_ida_volta"]
        j = cb["total"].idxmin()
        cb2 = cb.assign(mes=pd.to_datetime(cb["data_voo_ida"]).dt.to_period("M"))
        tabela = ["Mês     Melhor     Saída  Volta", "─" * 32]
        for mes, g in cb2.groupby("mes"):
            m = g.loc[g["total"].idxmin()]
            marca = " ✓" if m["total"] <= teto else ""
            tabela.append(f"{MESES[mes.month - 1]}/{str(mes.year)[2:]}  {brl(m['total']):>9}  "
                          f"{fmt_data(m['data_voo_ida'], False)}  {fmt_data(m['data_voo_volta'], False)}{marca}")
        ev = evolucao_por_mes(rota)
        tendencia = ""
        if not ev.empty and ev["dia"].nunique() >= 2:
            por_dia = ev.groupby("dia")["total"].min().sort_index()
            dif = por_dia.iloc[-1] - por_dia.iloc[-2]
            tendencia = (f"\n📈 Em relação à busca anterior ({por_dia.index[-2]:%d/%m}): melhor total "
                         + ("estável" if abs(dif) < 10 else f"{'caiu' if dif < 0 else 'subiu'} {brl(abs(dif))}") + ".")
        tab = tabelas[nome]
        ida_min = tab[tab["sentido"] == "ida"].nsmallest(1, "preco")
        vol_min = tab[tab["sentido"] == "volta"].nsmallest(1, "preco")
        resumo = (f"📊 <b>Resumo diário — {nome}</b>\n{cab}\n\n"
                  f"Melhor ida+volta: <b>{brl(cb.loc[j, 'total'])}</b> ({fmt_data(cb.loc[j, 'data_voo_ida'])} → "
                  f"{fmt_data(cb.loc[j, 'data_voo_volta'])}, {int(cb.loc[j, 'noites'])} noites) — "
                  + (f"{brl(teto - cb.loc[j, 'total'])} abaixo do teto." if cb.loc[j, 'total'] <= teto
                     else f"{brl(cb.loc[j, 'total'] - teto)} acima do teto.")
                  + f"\nDatas de saída dentro do teto: <b>{int((cb['total'] <= teto).sum())}</b> de {len(cb)}."
                  + (f"\nIda mais barata: {brl(ida_min['preco'].iloc[0])} em {fmt_data(ida_min['data_voo'].iloc[0])} "
                     f"({ida_min['cia'].iloc[0]})" if len(ida_min) else "")
                  + (f"\nVolta mais barata: {brl(vol_min['preco'].iloc[0])} em {fmt_data(vol_min['data_voo'].iloc[0])} "
                     f"({vol_min['cia'].iloc[0]})" if len(vol_min) else "")
                  + tendencia
                  + "\n\n<b>Melhor ida+volta por mês de saída</b> (✓ = dentro do teto)\n<pre>"
                  + html.escape("\n".join(tabela)) + "</pre>")
        tg_texto(resumo)
        if nome in paineis:
            tg_foto(paineis[nome], f"📊 Painel — {html.escape(nome)} · {AGORA:%d/%m %H:%M}\n"
                                    "A: preço por dia do voo · B: ida+volta vs teto · C: tendência por mês da viagem")

if not DRY_RUN:   # em teste não marca como enviado, para o alerta real ainda sair
    hist.gravar_alertas(pd.DataFrame(registro_alertas, columns=COLS_ALERTAS))
print(f"\n{len(registro_alertas)} oferta(s) alertada(s) nesta execução.")


In [ ]:
# CÉLULA 11 — REGISTRO DA EXECUÇÃO E "ÚLTIMO RESULTADO"
# execucoes.csv: 1 linha por execução e rota (bom para ver no GitHub se o robô está saudável).
# dados/ultimo/: painel e tabela da última varredura completa, para abrir direto no GitHub.
for rota in ROTAS:
    nome = rota["nome"]
    est = [estatisticas.get((nome, s), {}) for s in ("ida", "volta")]
    cb = combos.get(nome, pd.DataFrame())
    hist.registrar_execucao(dict(
        data_busca=AGORA.isoformat(timespec="seconds"),
        rota=nome,
        modo=modos.get(nome),
        fonte=est[0].get("fonte"),
        consultas=sum(e.get("consultas", 0) for e in est),
        sem_voo=sum(e.get("sem_voo", 0) for e in est),
        falhas=sum(e.get("falhas", 0) for e in est),
        melhor_ida_volta=None if cb.empty else round(float(cb["total"].min()), 2),
        datas_no_teto=0 if cb.empty else int((cb["total"] <= rota["teto_ida_volta"]).sum()),
        alertas_enviados=sum(1 for a in registro_alertas if a["rota"] == nome),
        dry_run=DRY_RUN,
        duracao_seg=round((datetime.now(TZ) - AGORA).total_seconds()),
    ))
    if modos.get(nome) == "completo" and not cb.empty:
        i = ROTAS.index(rota)
        cols = ["data_voo_ida", "data_voo_volta", "noites", "total", "preco_ida", "preco_volta",
                "origem_ida", "destino_ida", "cia_ida", "partida_ida", "origem_volta", "destino_volta",
                "cia_volta", "partida_volta", "link_ida", "link_volta"]
        hist.salvar_ultimo(f"melhor_ida_volta_por_data_{i}.csv", conteudo_df=cb[cols])
        if nome in paineis:
            hist.salvar_ultimo(f"painel_{i}.png", arquivo_origem=paineis[nome])

print(f"Execução {ID_EXECUCAO} registrada em {PASTA_DADOS}/execucoes.csv. [CONCLUÍDO]")


## Notas e limites

- **Custo de consultas:** a varredura completa faz ~1 consulta por data e sentido (≈ 360 por rota). O modo rápido faz ≈ 60. Se o Google começar a devolver páginas vazias, o notebook manda um aviso de “Monitor com problema” no Telegram — aí aumente `PAUSA_SEG` ou reduza `WORKERS`.
- **Ida e volta separadas:** o preço é a soma de dois bilhetes só ida. Quase sempre é igual ou mais barato que ida+volta fechado no mercado doméstico, mas são duas reservas.
- **Aviasales é cache:** preços dessa fonte vêm da busca de outros usuários nas últimas ~48 h; aparecem marcados como “confirme”.
- **Bagagem:** os preços são da tarifa mais barata (normalmente só bagagem de mão).
- **Testar sem internet:** rode com `FONTE="simulada"` e `DRY_RUN=1` — as mensagens aparecem impressas aqui em vez de ir para o Telegram.
